# CASMI26 v6: cosine floor + GBM fills
CPU-only, offline. Writes `submission.csv`.

In [ ]:
"""Subformula labelling (MIST-CF lite, RDKit-free).
Assign each MS2 peak a subformula of the candidate precursor formula.
RDBE filter, ppm matching, adduct-adjusted masses.
"""
import numpy as np
from itertools import product

# monoisotopic masses
ELEM_MASS = {
    "C": 12.0, "H": 1.00782503223, "N": 14.00307400443, "O": 15.99491461957,
    "P": 30.9737619985, "S": 31.9720711744, "F": 18.99840316273,
    "Cl": 34.968852682, "Br": 78.9183376, "I": 126.9044719,
    "Na": 22.9897692809, "K": 38.9637074864,
}
ELEM_ORDER = ["C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"]

ADDUCT_DELTA = {
    "[M+H]+": 1.007276, "[M+Na]+": 22.989218, "[M+K]+": 38.963158,
    "[M+NH4]+": 18.033823, "[M-H]-": -1.007276, "[M+Cl]-": 34.968853,
    "[M+CH2O2-H]-": 44.998201, "[M+C2H4O2-H]-": 59.013851,
    "[M]+": 0.0, "[M-H2O+H]+": -17.003348, "[M-2H2O+H]+": -35.013913,
    "[2M+H]+": None, "[2M+Na]+": None, "[2M-H]-": None, "[M+2H]2+": None,
}


def parse_formula(s):
    """'C9H8N2O2' -> dict. Handles two-letter elements."""
    import re
    out = {}
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", s):
        if el not in ELEM_MASS:
            return None
        out[el] = out.get(el, 0) + (int(n) if n else 1)
    return out


def formula_mass(f):
    return sum(ELEM_MASS[e] * n for e, n in f.items())


def rdbe(f):
    """Ring-double-bond equivalents. None if elements unsupported."""
    c = f.get("C", 0); h = f.get("H", 0); n = f.get("N", 0)
    hal = sum(f.get(e, 0) for e in ("F", "Cl", "Br", "I"))
    for e in f:
        if e not in ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"):
            return None
    return c - (h + hal) / 2 + n / 2 + 1


def enumerate_subformulae(prec_f, max_n=200000):
    """All f ⊆ prec_f with RDBE >= 0, as (counts_tuple, mass). Bounded."""
    keys = [e for e in ELEM_ORDER if e in prec_f]
    counts = [prec_f[e] for e in keys]
    # guard combinatorial explosion (e.g. C30H50...): cap by sampling coarse grid
    total = 1
    for c in counts:
        total *= (c + 1)
    subs = []
    if total <= max_n:
        for combo in product(*[range(c + 1) for c in counts]):
            if all(v == 0 for v in combo):
                continue
            f = dict(zip(keys, combo))
            if (rdbe(f) or -1) < 0:
                continue
            subs.append((combo, sum(ELEM_MASS[e] * n for e, n in zip(keys, combo))))
    else:
        # vectorized random sampling for huge combinatorial spaces
        rng = np.random.default_rng(0)
        k = len(keys)
        cm = np.array(counts)
        draws = rng.integers(0, cm + 1, size=(min(max_n * 3, 600000), k))
        draws = np.unique(draws, axis=0)
        nz = draws[np.any(draws > 0, axis=1)][:max_n]
        idx = {e: i for i, e in enumerate(keys)}
        hal_cols = [idx[e] for e in ("F", "Cl", "Br", "I") if e in idx]
        hal = nz[:, hal_cols].sum(axis=1) if hal_cols else 0
        c = nz[:, idx["C"]] if "C" in idx else 0
        h = nz[:, idx["H"]] if "H" in idx else 0
        n = nz[:, idx["N"]] if "N" in idx else 0
        ok = (c - (h + hal) / 2 + n / 2 + 1) >= 0
        sup = ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I")
        if any(e not in sup for e in keys):
            ok = ok & False
        nz = nz[ok][:max_n]
        mv = np.array([ELEM_MASS[e] for e in keys])
        masses = nz @ mv
        subs = [(tuple(row), float(m)) for row, m in zip(nz.tolist(), masses.tolist())]
    return keys, subs


_SUB_CACHE = {}


def subformula_masses(prec_formula_str, max_n=200000):
    """Cached (keys, masses array) for a precursor formula."""
    hit = _SUB_CACHE.get(prec_formula_str)
    if hit is not None:
        return hit
    prec_f = parse_formula(prec_formula_str)
    if prec_f is None:
        return None
    keys, subs = enumerate_subformulae(prec_f, max_n)
    arr = np.array([m for _, m in subs], dtype=float)
    _SUB_CACHE[prec_formula_str] = (keys, arr)
    return keys, arr


def label_peaks(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Greedy: for each top-N peak (by intensity), nearest subformula mass within ppm.
    Returns list of (mz, intensity, subformula_mass or None, ppm_err or None).
    Assumes fragments carry precursor adduct (MIST-CF assumption).
    """
    cached = subformula_masses(prec_formula_str)
    if cached is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    d = ADDUCT_DELTA.get(adduct)
    if d is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    mzs = np.asarray(mzs, dtype=float); intens = np.asarray(intens, dtype=float)
    order = np.argsort(-intens)[:top_n]
    _, sub_masses = cached
    out = []
    for idx in order:
        target = mzs[idx] - d  # adduct-adjusted neutral fragment mass
        if len(sub_masses) == 0:
            out.append((mzs[idx], intens[idx], None, None))
            continue
        j = int(np.argmin(np.abs(sub_masses - target)))
        err_ppm = abs(sub_masses[j] - target) / max(target, 1e-9) * 1e6
        if err_ppm <= ppm:
            out.append((mzs[idx], intens[idx], float(sub_masses[j]), float(err_ppm)))
        else:
            out.append((mzs[idx], intens[idx], None, None))
    return out


def explained_intensity(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Fraction of top-N intensity explained by subformulae. Core v1 feature."""
    labelled = label_peaks(mzs, intens, prec_formula_str, adduct, ppm, top_n)
    tot = sum(i for _, i, _, _ in labelled)
    exp = sum(i for _, i, m, _ in labelled if m is not None)
    n_hit = sum(1 for _, _, m, _ in labelled if m is not None)
    return (exp / tot if tot > 0 else 0.0), n_hit


In [ ]:
"""v2: memory-based fingerprint prediction (MIST retrieval path, no training).
Query spectra -> cosine neighbors among mass-window train spectra ->
cosine-weighted fingerprint blend -> rank candidates by Tanimoto.
"""
import numpy as np, pandas as pd
from bisect import bisect_left, bisect_right

PROJECT = "."  # unused in kernel


def neutral_mass(prec, adduct):
    if adduct == "[2M+H]+": return (prec - 1.007276) / 2
    if adduct == "[2M+Na]+": return (prec - 22.989218) / 2
    if adduct == "[2M-H]-": return (prec + 1.007276) / 2
    d = ADDUCT_DELTA.get(adduct)
    return prec - d if d is not None else np.nan


def cosine(mz1, it1, mz2, it2, tol=0.02):
    a = np.asarray(it1, dtype=float); b = np.asarray(it2, dtype=float)
    na = float(np.sqrt((a * a).sum())); nb = float(np.sqrt((b * b).sum()))
    if na == 0 or nb == 0: return 0.0
    m1 = np.asarray(mz1, dtype=float); m2 = np.asarray(mz2, dtype=float)
    o1 = np.argsort(m1); o2 = np.argsort(m2)
    m1, a = m1[o1], a[o1]; m2, b = m2[o2], b[o2]
    i = j = 0; num = 0.0
    while i < len(m1) and j < len(m2):
        d = m1[i] - m2[j]
        if abs(d) <= tol: num += a[i] * b[j]; i += 1; j += 1
        elif d < 0: i += 1
        else: j += 1
    return num / (na * nb)


def tanimoto(a, b):
    inter = float(np.logical_and(a, b).sum())
    union = float(np.logical_or(a, b).sum())
    return inter / union if union > 0 else 0.0


def load_fp():
    df = pd.read_parquet(f"{PROJECT}/data/fingerprints.parquet")
    return {s: np.unpackbits(np.asarray(f, dtype=np.uint8)) for s, f in zip(df["smiles"], df["fp"])}


def run(n_query=30, seed=1, k_blend=15, ppm=20, min_n=200):
    rng = np.random.default_rng(seed)
    fp = load_fp()
    train = pd.read_parquet(f"{PROJECT}/data/train.parquet",
        columns=["normalized_smiles", "inchikey14", "adduct", "precursor_mz",
                 "ms2_mzs", "ms2_normalized_intensities"])
    train["neutral"] = [neutral_mass(p, a) for p, a in zip(train["precursor_mz"], train["adduct"])]
    train = train[np.isfinite(train["neutral"].values)]
    groups = np.array(train["inchikey14"].unique())
    held = set(rng.choice(groups, size=min(500, len(groups) // 20), replace=False))
    qpool = train[train["inchikey14"].isin(held)]
    structs = np.array(qpool["normalized_smiles"].unique())
    rng.shuffle(structs)
    queries = list(structs[:n_query])

    db = train[~train["inchikey14"].isin(held)]
    struct = db.groupby("normalized_smiles")["neutral"].median().reset_index()
    truth = qpool.groupby("normalized_smiles")["neutral"].median().reset_index()
    struct = pd.concat([struct, truth]).drop_duplicates("normalized_smiles")
    struct = struct.sort_values("neutral").reset_index(drop=True)
    masses = struct["neutral"].values
    smi = struct["normalized_smiles"].values
    # sampled db spectra per structure for neighbor search
    db_samp = db.groupby("normalized_smiles").head(3)

    hits = {1: 0, 5: 0, 25: 0}
    rr = []
    for qi, qs in enumerate(queries):
        qspec = qpool[qpool["normalized_smiles"] == qs]
        qmass = float(np.median([neutral_mass(p, a) for p, a in zip(qspec["precursor_mz"], qspec["adduct"])]))
        tol = qmass * ppm / 1e6
        lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        pcur = ppm
        while hi - lo < min_n and pcur < 500:
            pcur *= 2; tol = qmass * pcur / 1e6
            lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        cands = list(smi[lo:hi][:2000])
        win = db_samp[db_samp["normalized_smiles"].isin(set(cands))]
        # neighbor similarities: best cosine of each db spectrum vs query spectra
        sims = []
        for _, r in qspec.iterrows():
            for _, t in win.iterrows():
                c = cosine(r["ms2_mzs"], r["ms2_normalized_intensities"],
                           t["ms2_mzs"], t["ms2_normalized_intensities"])
                if c > 0.01:
                    sims.append((c, t["normalized_smiles"]))
        sims.sort(reverse=True)
        # blend top-k distinct neighbor fingerprints
        seen, num, den = set(), None, 0.0
        for c, s in sims:
            if s in seen: continue
            seen.add(s)
            f = fp.get(s)
            if f is None: continue
            num = c * f if num is None else num + c * f
            den += c
            if len(seen) >= k_blend: break
        pred = (num / den) if den > 0 else None
        scored = []
        for s in cands:
            f = fp.get(s)
            if f is None or pred is None: t = 0.0
            else: t = tanimoto(pred > 0.3, f)
            scored.append((t, s))
        scored.sort(reverse=True)
        rank = next((i + 1 for i, (_, s) in enumerate(scored) if s == qs), 10**9)
        rr.append(1 / rank if rank <= 25 else 0.0)
        for k in hits:
            if rank <= k: hits[k] += 1
        if (qi + 1) % 10 == 0: print(f"{qi+1}/{n_query} MRR25={np.mean(rr):.3f}", flush=True)
    print(f"n={n_query} hit@1={hits[1]/n_query:.3f} hit@5={hits[5]/n_query:.3f} hit@25={hits[25]/n_query:.3f} MRR@25={np.mean(rr):.3f}")




In [ ]:
"""v4 channels: spectral-entropy similarity + analog propagation (RDKit-free runtime).
Entropy similarity (Li et al. 2021, matchms): low-entropy (few dominant peaks)
spectra match more selectively than cosine.
Analog propagation: score spectrum-less candidates (COCONUT) via spectrally
similar analogs WITH spectra: max_a sim(query,a)^p * Tanimoto(cand, analog).
"""
import numpy as np


def clean(mz, it, n_max=500, noise_pct=0.01):
    mz = np.asarray(mz, dtype=float); it = np.asarray(it, dtype=float)
    if len(mz) == 0: return mz, it
    keep = it >= it.max() * noise_pct
    mz, it = mz[keep], it[keep]
    if len(mz) > n_max:
        o = np.argsort(-it)[:n_max]
        mz, it = mz[o], it[o]
    o = np.argsort(mz)
    return mz[o], it[o]


def _match(mz1, it1, mz2, it2, tol=0.02):
    """Greedy matched peak pairs. Returns (a_matched, b_matched) intensity arrays."""
    i = j = 0
    A, B = [], []
    while i < len(mz1) and j < len(mz2):
        d = mz1[i] - mz2[j]
        if abs(d) <= tol:
            A.append(it1[i]); B.append(it2[j]); i += 1; j += 1
        elif d < 0: i += 1
        else: j += 1
    return np.array(A), np.array(B)


def _entropy(it):
    s = it.sum()
    if s <= 0 or len(it) == 0: return 0.0
    p = it / s
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


def entropy_similarity(mz1, it1, mz2, it2, tol=0.02):
    """1 - (2*H_merged - H1 - H2)/ln(4) over matched peaks. 0..~1."""
    m1, i1 = clean(mz1, it1)
    m2, i2 = clean(mz2, it2)
    A, B = _match(m1, i1, m2, i2, tol)
    if len(A) < 3:
        return 0.0
    M = A + B
    s = 1.0 - (2 * _entropy(M) - _entropy(A) - _entropy(B)) / np.log(4)
    return float(max(0.0, min(1.0, s)))


def tanimoto(a, b):
    a = np.asarray(a, dtype=bool); b = np.asarray(b, dtype=bool)
    inter = float(np.logical_and(a, b).sum())
    union = float(np.logical_or(a, b).sum())
    return inter / union if union > 0 else 0.0


In [ ]:
"""v6 production: cosine top-5 floor + GBM-ranked fills.
Writes submission.csv. Needs v5/gbm_ranker.pkl in FP dir + sklearn.
"""
import numpy as np, pandas as pd, os, pickle
from bisect import bisect_left, bisect_right

import glob as _glob
_comp = _glob.glob("/kaggle/input/**/test.parquet", recursive=True)
_fp = _glob.glob("/kaggle/input/**/coconut_fp.parquet", recursive=True)
IN = __import__("os").path.dirname(_comp[0]) if _comp else "data"
FP = __import__("os").path.dirname(_fp[0]) if _fp else IN
OUT = "/kaggle/working" if _comp else "."
print("IN=", IN, "FP=", FP, "OUT=", OUT)
TOP_N = 150
P_POW, DM_MAX = 3.0, 200.0
FEATS = ["cos", "ent", "ana"]


def neutral_mass(prec, adduct):
    if adduct == "[2M+H]+": return (prec - 1.007276) / 2
    if adduct == "[2M+Na]+": return (prec - 22.989218) / 2
    if adduct == "[2M-H]-": return (prec + 1.007276) / 2
    d = ADDUCT_DELTA.get(adduct)
    return prec - d if d is not None else np.nan


def topn(mz, it, n=TOP_N):
    mz = np.asarray(mz, dtype=float); it = np.asarray(it, dtype=float)
    if len(mz) <= n: return mz, it
    o = np.argsort(-it)[:n]
    return mz[o], it[o]


def window(masses, smi, qmass, ppm=20, min_n=200, cap=2000):
    tol = qmass * ppm / 1e6
    lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
    pcur = ppm
    while hi - lo < min_n and pcur < 500:
        pcur *= 2; tol = qmass * pcur / 1e6
        lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
    return list(smi[lo:hi][:cap])


def main():
    with open(f"{FP}/gbm_ranker.pkl", "rb") as f:
        ranker = pickle.load(f)["model"]
    test = pd.read_parquet(f"{IN}/test.parquet")
    test["neutral"] = [neutral_mass(p, a) for p, a in zip(test["precursor_mz"], test["adduct"])]
    mol_neutral = test.groupby("molecule_id")["neutral"].median()
    train = pd.read_parquet(f"{IN}/train.parquet",
        columns=["normalized_smiles", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"])
    train["neutral"] = [neutral_mass(p, a) for p, a in zip(train["precursor_mz"], train["adduct"])]
    train = train[np.isfinite(train["neutral"].values)]
    tstruct = train.groupby("normalized_smiles")["neutral"].median()
    tmass = tstruct.sort_values().values
    tsmi = tstruct.sort_values().index.values
    tsamp = train.groupby("normalized_smiles").head(2).reset_index(drop=True)
    tneut = tsamp["neutral"].values

    cf = pd.read_parquet(f"{FP}/coconut_fp.parquet")
    cfp = {s: np.unpackbits(np.asarray(f, dtype=np.uint8)) for s, f in zip(cf["canonical_smiles"], cf["fp"])}
    co = cf.sort_values("exact_molecular_weight").reset_index(drop=True)
    cmass = co["exact_molecular_weight"].values
    csmi = co["canonical_smiles"].values
    tf = pd.read_parquet(f"{FP}/fingerprints.parquet")
    scol = "normalized_smiles" if "normalized_smiles" in tf.columns else "smiles"
    tfp = {s: np.unpackbits(np.asarray(f, dtype=np.uint8)) for s, f in zip(tf[scol], tf["fp"])}

    rows = []
    for mi, (mol, spectra) in enumerate(test.groupby("molecule_id")):
        qmass = float(mol_neutral.loc[mol])
        tcands = window(tmass, tsmi, qmass)
        twin = tsamp[tsamp["normalized_smiles"].isin(set(tcands))]
        qspecs = [topn(np.asarray(mz, dtype=float), np.asarray(it, dtype=float))
                  for mz, it in zip(spectra["ms2_mzs"], spectra["ms2_normalized_intensities"])]
        cspec, espec = {}, {}
        for _, r in spectra.iterrows():
            for _, t in twin.iterrows():
                e = entropy_similarity(r["ms2_mzs"], r["ms2_normalized_intensities"],
                                       t["ms2_mzs"], t["ms2_normalized_intensities"])
                c = cosine(*topn(np.asarray(r["ms2_mzs"], dtype=float),
                                 np.asarray(r["ms2_normalized_intensities"], dtype=float)),
                           *topn(np.asarray(t["ms2_mzs"], dtype=float),
                                 np.asarray(t["ms2_normalized_intensities"], dtype=float)))
                k = t["normalized_smiles"]
                if e > espec.get(k, 0): espec[k] = e
                if c > cspec.get(k, 0): cspec[k] = c
        cos_top5 = [s for _, s in sorted(((cspec.get(s, 0.0), s) for s in tcands), reverse=True)[:5]]
        dm = np.abs(tneut - qmass)
        pool = tsamp[dm <= DM_MAX]
        if len(pool) > 2000: pool = pool.sample(2000, random_state=mi)
        analogs = []
        for _, t in pool.iterrows():
            best = 0.0
            for qmz, qit in qspecs:
                e = entropy_similarity(qmz, qit, t["ms2_mzs"], t["ms2_normalized_intensities"])
                if e > best: best = e
            if best > 0.05: analogs.append((best, t["normalized_smiles"]))
        analogs.sort(reverse=True)
        analogs = analogs[:50]
        afps = [(s, tfp[s]) for _, s in analogs if s in tfp]
        asims = [a[0] for a in analogs[:len(afps)]]
        ccands = window(cmass, csmi, qmass, cap=3000)
        feats, order = [], []
        for s in tcands + ccands:
            f = tfp.get(s, cfp.get(s))
            ab = 0.0
            if f is not None:
                for sim, (_, g) in zip(asims, afps):
                    v = (sim ** P_POW) * tanimoto(f, g)
                    if v > ab: ab = v
            feats.append([cspec.get(s, 0.0), espec.get(s, 0.0), ab])
            order.append(s)
        proba = ranker.predict_proba(np.array(feats, dtype=np.float32))[:, 1]
        ranked = sorted(zip(proba, order), reverse=True)
        seen = set(cos_top5)
        out = list(cos_top5)
        for _, s in ranked:
            if s in seen: continue
            seen.add(s); out.append(s)
            if len(out) == 25: break
        rows.append((mol, ";".join(out[:25])))
        if (mi + 1) % 50 == 0: print(f"done {mi+1}/400", flush=True)
    pd.DataFrame(rows, columns=["molecule_id", "smiles"]).to_csv(f"{OUT}/submission.csv", index=False)
    print("wrote submission.csv")



main()
